# Quran Data Ingestion with Prefect

This notebook demonstrates how to fetch data from the [Al Quran Cloud API](https://alquran.cloud/api) and orchestrate the process using **Prefect**.

### Prerequisites
You will need `prefect` and `requests` installed.

In [ ]:
# !pip install prefect requests

In [ ]:
import requests
from prefect import task, flow, get_run_logger
from typing import List, Union, Optional
import json

BASE_URL = "http://api.alquran.cloud/v1/ayah"

## 1. Define Prefect Tasks

We will define a task to fetch an ayah. This task will handle both single and multiple editions.

In [ ]:
@task(retries=3, retry_delay_seconds=5)
def fetch_ayah(reference: str, editions: Optional[Union[str, List[str]]] = None):
    """
    Fetches an ayah for a given reference and one or more editions.
    
    Args:
        reference: Ayah number (e.g., "262") or surah:ayah (e.g., "2:255")
        editions: Single edition string (e.g., "en.asad") or list of editions.
    """
    logger = get_run_logger()
    
    if editions is None:
        # Default to Arabic text if no edition is provided
        url = f"{BASE_URL}/{reference}"
    elif isinstance(editions, str):
        # Single edition
        url = f"{BASE_URL}/{reference}/{editions}"
    else:
        # Multiple editions
        editions_str = ",".join(editions)
        url = f"{BASE_URL}/{reference}/editions/{editions_str}"
    
    logger.info(f"Fetching data from: {url}")
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        data = response.json()
        if data["code"] == 200:
            return data["data"]
        else:
            logger.error(f"API Error: {data['status']}")
            return None
    except Exception as e:
        logger.error(f"Request failed: {e}")
        raise

## 2. Define the Prefect Flow

The flow coordinates the tasks and provides a clean interface for data retrieval.

In [ ]:
@flow(name="Quran Data Retrieval Flow")
def quran_ingestion_flow(references: List[str], editions: List[str]):
    """
    A flow to retrieve multiple ayahs across multiple editions.
    """
    results = []
    for ref in references:
        ayah_data = fetch_ayah(ref, editions)
        if ayah_data:
            results.append(ayah_data)
    
    return results

## 3. Run Examples

Let's fetch Ayat Al-Kursi (2:255) in Arabic, English (Asad), and English (Pickthall).

In [ ]:
if __name__ == "__main__":
    # Example: Fetch Ayat Al-Kursi in multiple editions
    # reference 2:255 or 262
    refs = ["2:255"]
    edits = ["quran-uthmani", "en.asad", "en.pickthall"]
    
    data = quran_ingestion_flow(refs, edits)
    
    # Displaying the results
    if data:
        print(f"Successfully fetched {len(data)} ayah(s).")
        # The API returns a list of objects when multiple editions are requested
        for entry in data[0]:
            edition_name = entry['edition']['name']
            text_preview = entry['text'][:100]
            print(f"\nEdition: {edition_name}\nText: {text_preview}...")